### Celda 6. Construir Gold: métricas del catálogo
Ahora calcularemos indicadores por género. Esta capa estará preparada para análisis posteriores y para alimentar el recomendador musical.

In [15]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

spark = (
    SparkSession.builder
    .appName("MusicDiscoveryLab-01")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

BASE_PATH = Path("/home/jovyan/data")

PATHS = {
    "raw": BASE_PATH / "raw",
    "bronze": BASE_PATH / "bronze" / "songs",
    "silver": BASE_PATH / "silver" / "songs",
    "gold": BASE_PATH / "gold" / "genre_metrics",
    "quarantine": BASE_PATH / "quarantine" / "songs",
}

raw_file = PATHS["raw"] / "songs.csv"

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Directorio de datos:", BASE_PATH)
print(f"Archivo entrada: {raw_file}")

Spark: 4.2.0
Master: local[*]
Directorio de datos: /home/jovyan/data
Archivo entrada: /home/jovyan/data/raw/songs.csv


In [16]:
bronze_df = spark.read.parquet(str(PATHS["bronze"]))
silver_df = spark.read.parquet(str(PATHS["silver"]))
quarantine_df = spark.read.parquet(str(PATHS["quarantine"]))

gold_df = (
    silver_df
    .groupBy("genre")
    .agg(
        F.count("*").alias("song_count"),
        F.round(F.avg("bpm"), 2).alias("avg_bpm"),
        F.round(F.avg("energy"), 3).alias("avg_energy"),
        F.round(F.avg("valence"), 3).alias("avg_valence"),
        F.round(
            F.avg("instrumentalness"), 3
        ).alias("avg_instrumentalness"),
    )
    .withColumn(
        "instrumental_profile",
        F.when(
            F.col("avg_instrumentalness") >= 0.8,
            F.lit("mostly_instrumental")
        ).otherwise(F.lit("mixed_or_vocal"))
    )
)

gold_df.write.mode("overwrite").parquet(str(PATHS["gold"]))

gold_df = spark.read.parquet(str(PATHS["gold"]))

gold_df.orderBy("genre").show(truncate=False)

+----------+----------+-------+----------+-----------+--------------------+--------------------+
|genre     |song_count|avg_bpm|avg_energy|avg_valence|avg_instrumentalness|instrumental_profile|
+----------+----------+-------+----------+-----------+--------------------+--------------------+
|Ambient   |3         |71.67  |0.257     |0.227      |0.963               |mostly_instrumental |
|Classical |3         |110.0  |0.6       |0.313      |0.95                |mostly_instrumental |
|Soundtrack|4         |113.5  |0.598     |0.39       |0.905               |mostly_instrumental |
+----------+----------+-------+----------+-----------+--------------------+--------------------+



In [17]:
quarantine_df.show(10)

+-------------+-------+------------+--------+----------+---+------+-------+----------------+----------------+---------------+
|source_row_id|song_id|       title|  artist|     genre|bpm|energy|valence|instrumentalness|rejection_reason|type_quarantine|
+-------------+-------+------------+--------+----------+---+------+-------+----------------+----------------+---------------+
|          010|   S010|        NULL|Artist J|   Ambient| 75|   0.3|    0.2|            0.95|   missing_title|  invalid_value|
|          011|   S011|Broken Tempo|Artist K|Soundtrack|350|   0.5|    0.4|            0.85|     invalid_bpm|  invalid_value|
+-------------+-------+------------+--------+----------+---+------+-------+----------------+----------------+---------------+



In [18]:
invalid_df = (
    quarantine_df
        .filter(F.col("type_quarantine") == "invalid_value")
)
invalid_df.show(10)

+-------------+-------+------------+--------+----------+---+------+-------+----------------+----------------+---------------+
|source_row_id|song_id|       title|  artist|     genre|bpm|energy|valence|instrumentalness|rejection_reason|type_quarantine|
+-------------+-------+------------+--------+----------+---+------+-------+----------------+----------------+---------------+
|          010|   S010|        NULL|Artist J|   Ambient| 75|   0.3|    0.2|            0.95|   missing_title|  invalid_value|
|          011|   S011|Broken Tempo|Artist K|Soundtrack|350|   0.5|    0.4|            0.85|     invalid_bpm|  invalid_value|
+-------------+-------+------------+--------+----------+---+------+-------+----------------+----------------+---------------+



### Celda 7. Verificar el resultado esperado
Para este conjunto de prueba, conocemos de antemano el resultado esperado. Eso nos permite comprobar que el pipeline funciona correctamente.

In [20]:
bronze_count = bronze_df.count()
invalid_count = invalid_df.count()
silver_count = silver_df.count()
quarantine_count = quarantine_df.count()

assert bronze_count == 12, "Cantidad inesperada en Bronze"
assert invalid_count == 2, "Se esperaban 2 registros inválidos"
assert silver_count == 10, "Se esperaban 10 canciones en Silver"
assert quarantine_count == 2, "Se esperaban 2 registros en cuarentena"

assert silver_df.select("song_id").distinct().count() == silver_count
assert silver_df.filter(
    (F.col("bpm") < 1) | (F.col("bpm") > 300)
).count() == 0

print("¡Todas las validaciones del laboratorio fueron satisfactorias!")
print({
    "bronze": bronze_count,
    "invalid_records": invalid_count,
    "silver": silver_count,
    "quarantine": quarantine_count,
    "gold_genres": gold_df.count(),
})

¡Todas las validaciones del laboratorio fueron satisfactorias!
{'bronze': 12, 'invalid_records': 2, 'silver': 10, 'quarantine': 2, 'gold_genres': 3}


### Celda 8. Leer Parquet de nuevo
Esta última prueba demuestra que el resultado persiste en disco y puede recuperarse incluso después de reiniciar el kernel.

In [21]:
silver_check = spark.read.parquet(str(PATHS["silver"]))
gold_check = spark.read.parquet(str(PATHS["gold"]))

silver_check.printSchema()
gold_check.show(truncate=False)

print("Silver persistido:", silver_check.count())
print("Gold persistido:", gold_check.count())

root
 |-- song_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- bpm: integer (nullable = true)
 |-- energy: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- instrumentalness: double (nullable = true)

+----------+----------+-------+----------+-----------+--------------------+--------------------+
|genre     |song_count|avg_bpm|avg_energy|avg_valence|avg_instrumentalness|instrumental_profile|
+----------+----------+-------+----------+-----------+--------------------+--------------------+
|Soundtrack|4         |113.5  |0.598     |0.39       |0.905               |mostly_instrumental |
|Ambient   |3         |71.67  |0.257     |0.227      |0.963               |mostly_instrumental |
|Classical |3         |110.0  |0.6       |0.313      |0.95                |mostly_instrumental |
+----------+----------+-------+----------+-----------+--------------------+--------------------+

Si